# CLIP4CAD-GFA v2.4 Training

**Shared Encoder Architecture**

## Key Fix from v2
v2 self-grounding collapsed because it had **too much capacity** (separate encoder).
It learned a different embedding space that didn't align with guided.

## v2.4 Solution: Share the Encoder
- **SHARED** grounding mechanism (same query projection for T_feat and Q_self)
- **SHARED** hierarchical aggregation
- Only `SelfQueryGenerator` is separate (minimal capacity)
- MSE embedding distillation (stronger gradient than cosine)
- Confidence alignment loss

## Training Stages
- **Stage 1 (Epochs 1-15)**: Heavy distillation to establish alignment
- **Stage 2 (Epochs 16-35)**: Balanced training with hard negatives

## Loss Weights
| Stage | λ_self | λ_distill | λ_embed (MSE) | λ_conf | λ_detail |
|-------|--------|-----------|---------------|--------|----------|
| 1 | 0.05 | 0.5 | **1.0** | 0.2 | 0.0 |
| 2 | 0.2 | 0.3 | **0.5** | 0.1 | 0.3 |

## Expected Results
- Self-cos: **0.85-0.95** (vs 0.15 in v2)
- Text→BRep R@1 (self): **60-65%** (vs 0.05% in v2)
- Text→BRep R@1 (guided): ~67% (maintained)

In [1]:
# Cell 1: Imports and Setup
import sys
sys.path.insert(0, '..')

import os
import gc
import math
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR
from tqdm.auto import tqdm
import numpy as np
from pathlib import Path

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Device: cuda
GPU: NVIDIA GeForce RTX 4090
Memory: 25.8 GB


In [2]:
# Cell 2: Data Paths
DATA_ROOT = Path("d:/Defect_Det/MMCAD/data")
PC_FILE = Path("c:/Users/User/Desktop/pc_embeddings_full.h5")
BREP_FILE = Path("c:/Users/User/Desktop/brep_features.h5")
TEXT_FILE = Path("c:/Users/User/Desktop/text_embeddings.h5")
OUTPUT_DIR = Path("../outputs/gfa_v2_4")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data root: {DATA_ROOT}")
print(f"PC file: {PC_FILE} (exists: {PC_FILE.exists()})")
print(f"BRep file: {BREP_FILE} (exists: {BREP_FILE.exists()})")
print(f"Text file: {TEXT_FILE} (exists: {TEXT_FILE.exists()})")
print(f"Output: {OUTPUT_DIR}")

Data root: d:\Defect_Det\MMCAD\data
PC file: c:\Users\User\Desktop\pc_embeddings_full.h5 (exists: True)
BRep file: c:\Users\User\Desktop\brep_features.h5 (exists: True)
Text file: c:\Users\User\Desktop\text_embeddings.h5 (exists: False)
Output: ..\outputs\gfa_v2_4


In [3]:
# Cell 3: Load Data
from clip4cad.data.gfa_dataset import GFAMappedDataset

print("Loading datasets...")
print("=" * 60)

# Train dataset - LOAD TO RAM for fast training
print("\n[1/2] Loading TRAIN dataset to RAM...")
train_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="train",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=True,
    use_live_text=False,
)
print(f"Train: {len(train_dataset):,} samples in RAM")

# Val dataset - ON DISK (saves RAM)
print("\n[2/2] Loading VAL dataset (on disk)...")
val_dataset = GFAMappedDataset(
    data_root=str(DATA_ROOT),
    split="val",
    pc_file=str(PC_FILE),
    text_file=str(TEXT_FILE),
    brep_file=str(BREP_FILE),
    num_rotations=1,
    load_to_memory=False,
    use_live_text=False,
)
print(f"Val: {len(val_dataset):,} samples on disk")

print("\n" + "=" * 60)
print("DATASETS READY!")

Loading datasets...

[1/2] Loading TRAIN dataset to RAM...
  Loading train data to memory (B-Rep + PC + Text)...
    Loading B-Rep (3GB)...
    Loading PC (50GB)...
    Loading Text from: c:\Users\User\Desktop\text_splits\train_text_embeddings.h5
    ✓ Text loaded: 174.6GB in 358.7s
    ⚠️  Pre-split has 111000 samples, dataset expected 133105
    Using first 111000 samples to match pre-split file
  ✓ Loaded 111000 samples: 203.6GB in RAM (B-Rep + PC + Text)
GFAMappedDataset: train with 111000 samples (in memory)
Train: 111,000 samples in RAM

[2/2] Loading VAL dataset (on disk)...
GFAMappedDataset: val with 16638 samples
Val: 16,638 samples on disk

DATASETS READY!


In [4]:
# Cell 4: Verify Dataset
sample = train_dataset[0]
print(f"Sample keys: {list(sample.keys())}")
print(f"  brep_face_features: {sample['brep_face_features'].shape}")
print(f"  brep_edge_features: {sample['brep_edge_features'].shape}")
print(f"  pc_features: {sample['pc_features'].shape}")
print(f"  desc_embedding: {sample['desc_embedding'].shape}")

Sample keys: ['sample_id', 'idx', 'rot_idx', 'brep_face_features', 'brep_edge_features', 'brep_face_mask', 'brep_edge_mask', 'pc_features', 'desc_embedding', 'desc_mask', 'has_brep', 'use_cached_brep_features', 'has_pointcloud', 'use_cached_pc_features', 'has_text', 'use_cached_embeddings']
  brep_face_features: torch.Size([192, 48])
  brep_edge_features: torch.Size([512, 12])
  pc_features: torch.Size([48, 1024])
  desc_embedding: torch.Size([256, 3072])


In [5]:
# Cell 5: Create Model
from clip4cad.models import CLIP4CAD_GFA_v2_4, GFAv2_4Config
from clip4cad.losses import GFAv2_4Loss
from clip4cad.losses.gfa_v2_4_losses import compute_self_grounding_quality

# Configuration
config = GFAv2_4Config(
    d_face=48,
    d_edge=12,
    d_pc=1024,
    d_text=3072,
    d_unified=256,
    d_proj=128,
    d_ground=128,
    num_slots=12,
    num_detail_queries=8,
    num_heads=8,
    num_parser_layers=2,
    num_self_query_layers=2,
    dropout=0.1,
)

model = CLIP4CAD_GFA_v2_4(config).to(device)
print(f"Model: CLIP4CAD-GFA v2.4 (Shared Encoder)")
print(f"Parameters: {model.count_parameters():,}")
print(f"Trainable: {model.count_parameters(trainable_only=True):,}")

Model: CLIP4CAD-GFA v2.4 (Shared Encoder)
Parameters: 7,362,055
Trainable: 7,362,055


In [6]:
# Cell 6: Training Configuration
from clip4cad.data.gfa_dataset import gfa_collate_fn

# Hyperparameters
BATCH_SIZE = 512
STAGE1_EPOCHS = 15
STAGE2_EPOCHS = 20
STAGE1_LR = 3e-5
STAGE2_LR = 1e-5
WARMUP_EPOCHS = 3
MAX_GRAD_NORM = 1.0

# v2.4 Loss weights - Heavy distillation with MSE
# Stage 1: Focus on alignment (distillation-heavy)
STAGE1_LAMBDA_SELF = 0.05             # Very low - don't let self compete
STAGE1_LAMBDA_DISTILL = 0.5           # High - grounding alignment
STAGE1_LAMBDA_EMBED_DISTILL = 1.0     # Very high - MSE embedding alignment
STAGE1_LAMBDA_CONF_ALIGN = 0.2        # Match activation patterns
STAGE1_LAMBDA_DETAIL = 0.0

# Stage 2: Balanced
STAGE2_LAMBDA_SELF = 0.2
STAGE2_LAMBDA_DISTILL = 0.3
STAGE2_LAMBDA_EMBED_DISTILL = 0.5
STAGE2_LAMBDA_CONF_ALIGN = 0.1
STAGE2_LAMBDA_DETAIL = 0.3

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
    collate_fn=gfa_collate_fn,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=gfa_collate_fn,
)

print(f"Batch size: {BATCH_SIZE}")
print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Total epochs: {STAGE1_EPOCHS + STAGE2_EPOCHS}")
print(f"\nStage 1 weights: self={STAGE1_LAMBDA_SELF}, distill={STAGE1_LAMBDA_DISTILL}, embed_distill(MSE)={STAGE1_LAMBDA_EMBED_DISTILL}, conf_align={STAGE1_LAMBDA_CONF_ALIGN}")
print(f"Stage 2 weights: self={STAGE2_LAMBDA_SELF}, distill={STAGE2_LAMBDA_DISTILL}, embed_distill(MSE)={STAGE2_LAMBDA_EMBED_DISTILL}, conf_align={STAGE2_LAMBDA_CONF_ALIGN}")

Batch size: 512
Train batches: 216
Val batches: 33
Total epochs: 35

Stage 1 weights: self=0.05, distill=0.5, embed_distill(MSE)=1.0, conf_align=0.2
Stage 2 weights: self=0.2, distill=0.3, embed_distill(MSE)=0.5, conf_align=0.1


In [7]:
# Cell 7: Initialize Optimizer, Loss, Scheduler
from clip4cad.training.hard_negative_mining import HardNegativeMiner

optimizer = AdamW(
    model.parameters(),
    lr=STAGE1_LR,
    weight_decay=0.01,
    betas=(0.9, 0.999),
)

criterion = GFAv2_4Loss(
    lambda_self=STAGE1_LAMBDA_SELF,
    lambda_distill=STAGE1_LAMBDA_DISTILL,
    lambda_embed_distill=STAGE1_LAMBDA_EMBED_DISTILL,
    lambda_conf_align=STAGE1_LAMBDA_CONF_ALIGN,
    lambda_detail=STAGE1_LAMBDA_DETAIL,
)

scaler = GradScaler()

# Learning rate scheduler with warmup
total_epochs = STAGE1_EPOCHS + STAGE2_EPOCHS
warmup_steps = WARMUP_EPOCHS * len(train_loader)
total_steps = total_epochs * len(train_loader)

def lr_lambda(step):
    if step < warmup_steps:
        return step / max(warmup_steps, 1)
    progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
    return max(1e-6 / STAGE1_LR, 0.5 * (1 + math.cos(math.pi * progress)))

scheduler = LambdaLR(optimizer, lr_lambda)

# Hard negative miner (used in Stage 2)
hard_neg_miner = HardNegativeMiner(
    model=model,
    train_dataloader=train_loader,
    cache_dir=str(OUTPUT_DIR / "hard_negatives"),
    k=20,
    text_sim_threshold=0.8,
    min_negatives=1,
    max_negatives=10,
    use_faiss=True,
    device=str(device),
)
hard_negatives = None
MINE_EVERY_N_EPOCHS = 5

print("Optimizer, loss, scheduler, and hard negative miner initialized.")
print(f"Using MSE for embedding distillation (stronger gradient than cosine)")

Optimizer, loss, scheduler, and hard negative miner initialized.
Using MSE for embedding distillation (stronger gradient than cosine)


C:\Users\User\AppData\Local\Temp\ipykernel_15804\2328809447.py:19: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


In [8]:
# Cell 8: Training Loop

# Training state
global_step = 0
best_val_loss = float('inf')
best_self_cosine = 0.0
current_stage = 1

# Training history
history = {
    'train_loss': [],
    'val_loss': [],
    'self_cosine_brep': [],
    'self_cosine_pc': [],
    'embed_distill': [],
    'conf_align': [],
}

print("Starting training...")
print("="*70)

for epoch in range(1, total_epochs + 1):
    # Stage transition
    if epoch == STAGE1_EPOCHS + 1:
        print("\n" + "="*70)
        print("TRANSITIONING TO STAGE 2")
        print("="*70)
        current_stage = 2
        
        # Update loss weights
        criterion.update_weights(
            lambda_self=STAGE2_LAMBDA_SELF,
            lambda_distill=STAGE2_LAMBDA_DISTILL,
            lambda_embed_distill=STAGE2_LAMBDA_EMBED_DISTILL,
            lambda_conf_align=STAGE2_LAMBDA_CONF_ALIGN,
            lambda_detail=STAGE2_LAMBDA_DETAIL,
        )
        print(f"Updated loss weights: self={STAGE2_LAMBDA_SELF}, embed_distill={STAGE2_LAMBDA_EMBED_DISTILL}, conf_align={STAGE2_LAMBDA_CONF_ALIGN}, detail={STAGE2_LAMBDA_DETAIL}")
        
        # Reduce learning rate
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
        print(f"Reduced LR to {STAGE2_LR}")
        
        # Save Stage 1 checkpoint
        torch.save({
            'epoch': epoch - 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_self_cosine': best_self_cosine,
        }, OUTPUT_DIR / 'checkpoint_stage1_final.pt')
        print(f"Saved Stage 1 checkpoint")
        
        # Mine hard negatives at start of Stage 2
        print("\nMining hard negatives for Stage 2...")
        hard_negatives = hard_neg_miner.mine(epoch=epoch)
        print(f"Mined hard negatives for {len(hard_negatives)} samples")
    
    # Re-mine hard negatives every N epochs in Stage 2
    if current_stage == 2 and epoch > STAGE1_EPOCHS + 1:
        if (epoch - STAGE1_EPOCHS - 1) % MINE_EVERY_N_EPOCHS == 0:
            print(f"\nRe-mining hard negatives (epoch {epoch})...")
            hard_negatives = hard_neg_miner.mine(epoch=epoch)
            print(f"Re-mined hard negatives for {len(hard_negatives)} samples")
    
    # Train epoch
    model.train()
    epoch_loss = 0.0
    epoch_self_cos_brep = []
    epoch_self_cos_pc = []
    epoch_embed_distill = []
    epoch_conf_align = []
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} (Stage {current_stage})")
    for batch_idx, batch in enumerate(pbar):
        # Get hard negatives for this batch (if in Stage 2)
        batch_hard_negs = None
        if current_stage == 2 and hard_negatives is not None:
            batch_size = batch['brep_face_features'].shape[0]
            start_idx = batch_idx * BATCH_SIZE
            batch_hard_negs = []
            for i in range(batch_size):
                sample_idx = start_idx + i
                if sample_idx in hard_negatives:
                    batch_hard_negs.append(hard_negatives[sample_idx])
                else:
                    batch_hard_negs.append(None)
        
        with autocast():
            outputs = model(batch)
            loss, loss_dict = criterion(outputs, hard_negatives=batch_hard_negs, stage=current_stage)
        
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        
        global_step += 1
        epoch_loss += loss_dict['total']
        epoch_embed_distill.append(loss_dict.get('embed_distill', 0))
        epoch_conf_align.append(loss_dict.get('conf_align', 0))
        
        # Compute self-grounding quality
        if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
            cos_brep = compute_self_grounding_quality(
                outputs['z_brep'].detach(),
                outputs['z_brep_self'].detach()
            )
            epoch_self_cos_brep.append(cos_brep)
        
        if outputs.get('z_pc') is not None and outputs.get('z_pc_self') is not None:
            cos_pc = compute_self_grounding_quality(
                outputs['z_pc'].detach(),
                outputs['z_pc_self'].detach()
            )
            epoch_self_cos_pc.append(cos_pc)
        
        # Update progress bar
        postfix = {
            'loss': f"{loss_dict['total']:.3f}",
            'guided': f"{loss_dict['guided']:.3f}",
            'emb_d': f"{loss_dict.get('embed_distill', 0):.3f}",
            'conf': f"{loss_dict.get('conf_align', 0):.3f}",
            'cos': f"{epoch_self_cos_brep[-1]:.3f}" if epoch_self_cos_brep else "N/A",
        }
        if current_stage == 2:
            postfix['detail'] = f"{loss_dict.get('detail', 0):.3f}"
        pbar.set_postfix(postfix)
    
    # Epoch summary
    avg_loss = epoch_loss / len(train_loader)
    avg_cos_brep = sum(epoch_self_cos_brep) / len(epoch_self_cos_brep) if epoch_self_cos_brep else 0
    avg_cos_pc = sum(epoch_self_cos_pc) / len(epoch_self_cos_pc) if epoch_self_cos_pc else 0
    avg_embed_distill = sum(epoch_embed_distill) / len(epoch_embed_distill) if epoch_embed_distill else 0
    avg_conf_align = sum(epoch_conf_align) / len(epoch_conf_align) if epoch_conf_align else 0
    
    history['train_loss'].append(avg_loss)
    history['self_cosine_brep'].append(avg_cos_brep)
    history['self_cosine_pc'].append(avg_cos_pc)
    history['embed_distill'].append(avg_embed_distill)
    history['conf_align'].append(avg_conf_align)
    
    if avg_cos_brep > best_self_cosine:
        best_self_cosine = avg_cos_brep
    
    print(f"\nEpoch {epoch}: Loss={avg_loss:.4f}, Self-cos BRep={avg_cos_brep:.4f}, Embed-distill={avg_embed_distill:.4f}, Conf-align={avg_conf_align:.4f}")
    print(f"Best self-cosine: {best_self_cosine:.4f}")
    
    # Validation every 5 epochs
    if epoch % 5 == 0:
        model.eval()
        val_loss = 0.0
        val_cos_brep = []
        
        with torch.no_grad():
            for batch in tqdm(val_loader, desc="Validation"):
                with autocast():
                    outputs = model(batch)
                    loss, loss_dict = criterion(outputs, stage=current_stage)
                val_loss += loss_dict['total']
                
                if outputs.get('z_brep') is not None and outputs.get('z_brep_self') is not None:
                    cos_brep = compute_self_grounding_quality(
                        outputs['z_brep'],
                        outputs['z_brep_self']
                    )
                    val_cos_brep.append(cos_brep)
        
        avg_val_loss = val_loss / len(val_loader)
        avg_val_cos = sum(val_cos_brep) / len(val_cos_brep) if val_cos_brep else 0
        
        history['val_loss'].append(avg_val_loss)
        print(f"Validation: Loss={avg_val_loss:.4f}, Self-cos={avg_val_cos:.4f}")
        
        # Save best model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'best_val_loss': best_val_loss,
                'best_self_cosine': best_self_cosine,
            }, OUTPUT_DIR / 'checkpoint_best.pt')
            print("Saved best model!")
    
    # Save checkpoint every 5 epochs
    if epoch % 5 == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'best_self_cosine': best_self_cosine,
        }, OUTPUT_DIR / f'checkpoint_epoch{epoch}.pt')
    
    # Clear cache
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

print("\n" + "="*70)
print("Training Complete!")
print(f"Best self-grounding cosine: {best_self_cosine:.4f}")
print(f"Best validation loss: {best_val_loss:.4f}")
print("="*70)

Starting training...


Epoch 1 (Stage 1):   0%|          | 0/216 [00:00<?, ?it/s]

C:\Users\User\AppData\Local\Temp\ipykernel_15804\4031123860.py:89: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
c:\Users\User\anaconda3\envs\clip4cad\lib\site-packages\torch\optim\lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(



Epoch 1: Loss=8.2409, Self-cos BRep=0.8266, Embed-distill=0.0059, Conf-align=0.0015
Best self-cosine: 0.8266


Epoch 2 (Stage 1):   0%|          | 0/216 [00:00<?, ?it/s]

...


Epoch 34: Loss=5.0665, Self-cos BRep=0.1319, Embed-distill=0.0221, Conf-align=0.0459
Best self-cosine: 0.8266


Epoch 35 (Stage 2):   0%|          | 0/216 [00:00<?, ?it/s]


Epoch 35: Loss=5.0618, Self-cos BRep=0.1333, Embed-distill=0.0220, Conf-align=0.0460
Best self-cosine: 0.8266


Validation:   0%|          | 0/33 [00:00<?, ?it/s]

Validation: Loss=4.9170, Self-cos=0.0843
Saved best model!

Training Complete!
Best self-grounding cosine: 0.8266
Best validation loss: 4.9170


In [9]:
# Cell 9: Save Final Model
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config.__dict__,
    'best_self_cosine': best_self_cosine,
    'history': history,
}, OUTPUT_DIR / 'clip4cad_gfa_v2_4_final.pt')

print(f"Final model saved to {OUTPUT_DIR / 'clip4cad_gfa_v2_4_final.pt'}")

Final model saved to ..\outputs\gfa_v2_4\clip4cad_gfa_v2_4_final.pt


In [10]:
# Cell 10: Plot Training History
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Loss plot
axes[0, 0].plot(history['train_loss'], label='Train Loss')
if history['val_loss']:
    val_epochs = list(range(5, len(history['train_loss']) + 1, 5))[:len(history['val_loss'])]
    axes[0, 0].plot(val_epochs, history['val_loss'], 'o-', label='Val Loss')
axes[0, 0].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].set_title('Training Loss')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Self-grounding quality plot
axes[0, 1].plot(history['self_cosine_brep'], label='BRep Self-Cosine')
axes[0, 1].plot(history['self_cosine_pc'], label='PC Self-Cosine')
axes[0, 1].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[0, 1].axhline(y=0.9, color='g', linestyle=':', label='Target (0.9)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Cosine Similarity')
axes[0, 1].set_title('Self-Grounding Quality (Should be HIGH with v2.4!)')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Embedding distillation (MSE)
axes[1, 0].plot(history['embed_distill'], label='Embed Distill (MSE)')
axes[1, 0].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('MSE Loss')
axes[1, 0].set_title('Embedding Distillation (MSE)')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Confidence alignment
axes[1, 1].plot(history['conf_align'], label='Conf Align')
axes[1, 1].axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Stage Transition')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('MSE Loss')
axes[1, 1].set_title('Confidence Alignment')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'training_history_v2_4.png', dpi=150)
plt.show()

print(f"Plot saved to {OUTPUT_DIR / 'training_history_v2_4.png'}")

: 